<a href="https://colab.research.google.com/github/lyntos/RAG_AI_assistant/blob/main/RAG_AI_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Simple RAG in Python

We will build a very simple RAG pipeline using:

sentence-transformers → for embeddings

faiss → for fast vector search

(optional) OpenAI → for generating final answers

What is happening here?

User asks a question

Question is converted into an embedding

FAISS searches for the most similar document

That document is passed to the LLM

LLM generates a final, accurate answer

Example Output

Retrieved: Refund policy: You can return products within 7 days.

Answer: You can return products within 7 days as per the refund policy.

In [40]:
# OPTION A (Recommended for Colab): OpenAI-free RAG using HuggingFace (NO API KEY)

# This is the best fully working Colab RAG code.

#  1. Install dependencies

In [41]:
!pip install sentence-transformers faiss-cpu numpy

In [42]:
# 2. Import libraries
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [43]:
# 3. Knowledge base
documents = [
    "Refund policy: You can return products within 7 days.",
    "Delivery time is usually 3 to 5 business days.",
    "Customer support is available 24/7.",
    "You can track your order using the tracking ID.",
]

In [44]:
# 4. Create embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

doc_embeddings = model.encode(documents)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [45]:
# 5. Build FAISS index
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

In [46]:
# 6. Ask a question
query = "How can I return a product?"
query_embedding = model.encode([query])

In [47]:
# 7. Retrieve best match
k = 1
distances, indices = index.search(np.array(query_embedding), k)

retrieved_doc = documents[indices[0][0]]

print("Retrieved Context:\n", retrieved_doc)

Retrieved Context:
 Refund policy: You can return products within 7 days.


In [48]:
# 8. Generate answer (NO API — simple RAG response)

# Since Colab cannot use Ollama easily, we simulate LLM with context-based answer:

answer = f"""
Based on the context:
{retrieved_doc}

Answer:
{retrieved_doc}
"""

print(answer)


Based on the context:
Refund policy: You can return products within 7 days.

Answer:
Refund policy: You can return products within 7 days.

